# A network will fit random labels — and what that means

Deep learning models can memorise an arbitrary mapping. Generalization therefore cannot come from the model; it comes from structure in the data.

**Runs on:** CPU — about 4 minutes &nbsp;·&nbsp; **Slides:** [Chapter 5 — Fundamentals of Machine Learning](../../../course-web-slides/ch05/index.html) &nbsp;·&nbsp; **Section:** 02 — The nature of generalization

---

## Shuffling the labels

In [ ]:
import numpy as np
import keras
from keras import layers
from keras.datasets import mnist

(train_images, train_labels), _ = mnist.load_data()
train_images = train_images.reshape((60000, 28 * 28)).astype("float32") / 255

random_train_labels = train_labels[:]
np.random.shuffle(random_train_labels)

model = keras.Sequential([
    layers.Dense(512, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="rmsprop",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
h = model.fit(train_images, random_train_labels,
              epochs=100, batch_size=128, validation_split=0.2, verbose=0)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4.4))
plt.plot(h.history["accuracy"], label="training accuracy")
plt.plot(h.history["val_accuracy"], label="validation accuracy")
plt.axhline(0.1, color="k", ls=":", lw=1, label="chance (10 classes)")
plt.xlabel("epoch"); plt.legend()
plt.title("Fitting labels that carry no information at all")
plt.show()

print(f"final training accuracy:   {h.history['accuracy'][-1]:.3f}")
print(f"final validation accuracy: {h.history['val_accuracy'][-1]:.3f}")

Expected output:

```
final training accuracy:   0.9xx
final validation accuracy: 0.1xx
```

Training accuracy climbs toward 1.0. Validation accuracy sits at chance, because there is nothing to generalize to.

**Deep learning models can be trained to fit anything.** So generalization is not a property of the model — it is a property of *the structure of the data*, which the model can exploit when it exists and cannot invent when it does not.

## The manifold hypothesis, made visible

MNIST digits live in a 784-dimensional space. The claim is that they occupy a **very low-dimensional manifold** inside it. Here is one piece of evidence: a random point in that space, against a real digit.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))
for ax, img in zip(axes[0], train_images[:6]):
    ax.imshow(img.reshape(28, 28), cmap="gray_r"); ax.axis("off")
axes[0, 0].set_title("real digits", loc="left", fontsize=10)

for ax in axes[1]:
    ax.imshow(np.random.random((28, 28)), cmap="gray_r"); ax.axis("off")
axes[1, 0].set_title("uniform random points in the same space",
                     loc="left", fontsize=10)
plt.tight_layout(); plt.show()

Sampling uniformly from the 784-dimensional cube gives you noise, essentially always. **The digits occupy a vanishingly small region** — and it is a connected one, which is the part that matters next.

## Interpolating between two digits

In [ ]:
a = train_images[np.where(train_labels == 4)[0][0]]
b = train_images[np.where(train_labels == 9)[0][0]]

alphas = np.linspace(0, 1, 9)
fig, axes = plt.subplots(1, 9, figsize=(12, 1.8))
for ax, t in zip(axes, alphas):
    ax.imshow(((1 - t) * a + t * b).reshape(28, 28), cmap="gray_r")
    ax.set_title(f"{t:.2f}", fontsize=8); ax.axis("off")
plt.suptitle("Linear interpolation in pixel space", y=1.12)
plt.show()

The midpoints are **ghosts** — two digits superimposed, not a digit. Linear interpolation in *pixel* space leaves the manifold immediately.

Chapter 17 does the same interpolation in a **learned latent** space and every midpoint is a valid digit. That difference is the entire value of representation learning, and it is worth seeing the failure before seeing the success.

## Why more data is the best regularizer

In [ ]:
def train_on(n, epochs=20):
    keras.utils.set_random_seed(0)
    m = keras.Sequential([layers.Dense(512, activation="relu"),
                          layers.Dense(10, activation="softmax")])
    m.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    hh = m.fit(train_images[:n], train_labels[:n], epochs=epochs,
               batch_size=128, validation_split=0.2, verbose=0)
    return max(hh.history["val_accuracy"])

sizes = [500, 2000, 10000, 60000]
scores = [train_on(n) for n in sizes]
for n, s in zip(sizes, scores):
    print(f"{n:6d} samples -> best val acc {s:.4f}")

plt.figure(figsize=(6, 4))
plt.semilogx(sizes, scores, "o-")
plt.xlabel("training samples"); plt.ylabel("best validation accuracy")
plt.title("A denser sampling of the manifold generalizes better")
plt.show()

A model trained on a **dense** sampling of the manifold interpolates between points that are genuinely close together. On a sparse sampling it interpolates across gaps, and the interpolation is a guess. That is the whole argument.

---

## What to take away

- A network will fit shuffled labels to near-perfect training accuracy — so generalization comes from the **data**, not the model.
- Real data occupies a tiny, structured manifold inside its nominal space.
- Interpolating in pixel space leaves the manifold; interpolating in a learned space does not.
- More data is the most effective regularizer there is, because it samples the manifold more densely.